# Preprocessing for Machine Learning

Notebook này chuẩn bị dữ liệu feature-engineered cho bước ML theo spec. Mục tiêu chính là:

- xác định đúng input dataset và target theo feature engineering
- kiểm tra leakage và feature contract
- chia train/validation/test trước khi fit preprocessing
- áp dụng imputation + scaling cho numerical
- áp dụng imputation + one-hot encoding cho categorical
- lưu preprocessor để các model khác dùng lại cùng logic

Nguyên tắc đồng nhất với feature engineering:
- `Churned` chỉ là target, không bao giờ là feature đầu vào
- `City` được coi là excluded feature vì không nằm trong baseline feature set
- các feature target-derived bị cấm phải bị loại khỏi `X`
- tất cả tham số preprocessing đều học trên `X_train`, không trên toàn bộ dataset

## Section 01 — Load Feature-Engineered Dataset

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "Churned"

FORBIDDEN_FEATURES = [
    "Churn_Risk_Score",
    "Churn_Probability",
    "Churn_Level",
    "Churn_Status",
    "Churn_Risk",
    "Churned_Score",
]

EXCLUDED_FEATURES = ["City"]

PROJECT_ROOT = Path.cwd().resolve()
if Path.cwd().name == "notebooks" or not (PROJECT_ROOT / "data").exists():
    if (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "customer_churn_feature_engineered.csv"
MODELS_DIR = PROJECT_ROOT / "models"/"preprocessor"
MODELS_DIR.mkdir(exist_ok=True, parents=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"Excluded feature list: {EXCLUDED_FEATURES}")
print(f"Forbidden target-derived feature list: {FORBIDDEN_FEATURES}")

Project root: D:\Desktop\Projects\customer-churn-analytics
Dataset path: D:\Desktop\Projects\customer-churn-analytics\data\processed\customer_churn_feature_engineered.csv
Excluded feature list: ['City']
Forbidden target-derived feature list: ['Churn_Risk_Score', 'Churn_Probability', 'Churn_Level', 'Churn_Status', 'Churn_Risk', 'Churned_Score']


## Section 02 — Load Feature-Engineered Dataset

Tải dataset sau feature engineering để kiểm tra dữ liệu đầu vào. Đây là dataset đã qua feature engineering, nên contract của feature phải khớp với output đã export từ notebook 05.

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded from: {DATA_PATH}")
print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Target: {TARGET}")
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

assert TARGET in df.columns, f"Target column '{TARGET}' does not exist in the dataset."

Dataset loaded from: D:\Desktop\Projects\customer-churn-analytics\data\processed\customer_churn_feature_engineered.csv
Dataset shape: (50000, 33)
Rows: 50000
Columns: 33
Target: Churned

Columns:
['Gender', 'Country', 'Signup_Quarter', 'Age', 'Membership_Years', 'Login_Frequency', 'Session_Duration_Avg', 'Pages_Per_Session', 'Cart_Abandonment_Rate', 'Wishlist_Items', 'Total_Purchases', 'Average_Order_Value', 'Days_Since_Last_Purchase', 'Discount_Usage_Rate', 'Returns_Rate', 'Email_Open_Rate', 'Customer_Service_Calls', 'Product_Reviews_Written', 'Social_Media_Engagement_Score', 'Mobile_App_Usage', 'Payment_Method_Diversity', 'Lifetime_Value', 'Credit_Balance', 'Engagement_Score', 'Purchase_Frequency', 'Purchase_Recency_Group', 'Purchase_Frequency_Group', 'Service_Call_Group', 'Cart_Abandonment_Group', 'Estimated_Purchase_Value', 'Cart_Recency_Interaction', 'Low_Engagement_Recency', 'Churned']

Dtypes:
Gender                               str
Country                              str
Sign

## Section 03 — Data and target audit


In [3]:
target_distribution = df[TARGET].value_counts().sort_index()
print("Target distribution:")
print(target_distribution)
print("\nTarget percentage:")
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

print(f"\nTotal rows: {len(df)}")
print(f"Duplicate rows: {df.duplicated().sum()}")

missing_summary = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "feature", 0: "missing_count"})
)
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(2)
print("\nMissing values summary:")
print(missing_summary.sort_values("missing_count", ascending=False).to_string(index=False))

assert df[TARGET].notna().all(), "Target contains missing values."
assert set(df[TARGET].unique()).issubset({0, 1}), "Target is not binary {0, 1}."

Target distribution:
Churned
0    35550
1    14450
Name: count, dtype: int64

Target percentage:
Churned
0    71.1
1    28.9
Name: proportion, dtype: float64

Total rows: 50000
Duplicate rows: 0

Missing values summary:
                      feature  missing_count  missing_pct
                       Gender              0          0.0
                      Country              0          0.0
               Signup_Quarter              0          0.0
                          Age              0          0.0
             Membership_Years              0          0.0
              Login_Frequency              0          0.0
         Session_Duration_Avg              0          0.0
            Pages_Per_Session              0          0.0
        Cart_Abandonment_Rate              0          0.0
               Wishlist_Items              0          0.0
              Total_Purchases              0          0.0
          Average_Order_Value              0          0.0
     Days_Since_Last_Purch

## Section 04 — Leakage Audit

Theo feature engineering spec, `Churned` chỉ là target và không được có trong `X`. Ngoài ra, ta phải review các feature có nguy cơ leakage về mặt thời gian hoặc target-derived. Một số biến như `Days_Since_Last_Purchase`, `Total_Purchases`, `Lifetime_Value`, `Customer_Service_Calls`, `Credit_Balance` cần được xem lại dựa trên business context. Ở dataset hiện tại, chúng được giữ nếu là snapshot có sẵn trước khi dự đoán và bị loại nếu là hậu quả sau khi churn xảy ra.

In [4]:
potential_leakage_features = [
    "Days_Since_Last_Purchase",
    "Total_Purchases",
    "Lifetime_Value",
    "Customer_Service_Calls",
    "Credit_Balance",
]

print("Prediction-time availability review:")
for col in potential_leakage_features:
    if col in df.columns:
        print(
            f"- {col}: requires business confirmation that it is available before prediction time."
        )

excluded_features = [col for col in EXCLUDED_FEATURES if col in df.columns]
forbidden_features_present = [col for col in FORBIDDEN_FEATURES if col in df.columns]
if forbidden_features_present:
    raise ValueError(
        "Target-derived features are present in the engineered dataset: "
        + ", ".join(forbidden_features_present)
    )

print(f"Excluded features for model input: {excluded_features}")
print(f"Forbidden target-derived features present: {forbidden_features_present}")

assert TARGET in df.columns, f"Target column '{TARGET}' is missing from the feature-engineered dataset."

Prediction-time availability review:
- Days_Since_Last_Purchase: requires business confirmation that it is available before prediction time.
- Total_Purchases: requires business confirmation that it is available before prediction time.
- Lifetime_Value: requires business confirmation that it is available before prediction time.
- Customer_Service_Calls: requires business confirmation that it is available before prediction time.
- Credit_Balance: requires business confirmation that it is available before prediction time.
Excluded features for model input: []
Forbidden target-derived features present: []


## Section 05 — Define Modeling Features and Target

In [5]:
approved_features = [
    col
    for col in df.columns
    if col != TARGET
    and col not in excluded_features
    and col not in FORBIDDEN_FEATURES
]

df = df.reset_index(drop=True)
X = df[approved_features].copy()
y = df[TARGET].copy()

print(f"Target: {TARGET}")
print(f"Number of input features: {X.shape[1]}")
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")
print(f"\nApproved feature list ({len(approved_features)}):")
print(approved_features)

assert TARGET not in X.columns, "Target leakage: target column still present in input features."
assert len(approved_features) > 0, "No valid feature columns selected for modeling."
assert not set(FORBIDDEN_FEATURES).intersection(X.columns), "Forbidden target-derived features leaked into X."

Target: Churned
Number of input features: 32
Shape of X: (50000, 32)
Shape of y: (50000,)

Approved feature list (32):
['Gender', 'Country', 'Signup_Quarter', 'Age', 'Membership_Years', 'Login_Frequency', 'Session_Duration_Avg', 'Pages_Per_Session', 'Cart_Abandonment_Rate', 'Wishlist_Items', 'Total_Purchases', 'Average_Order_Value', 'Days_Since_Last_Purchase', 'Discount_Usage_Rate', 'Returns_Rate', 'Email_Open_Rate', 'Customer_Service_Calls', 'Product_Reviews_Written', 'Social_Media_Engagement_Score', 'Mobile_App_Usage', 'Payment_Method_Diversity', 'Lifetime_Value', 'Credit_Balance', 'Engagement_Score', 'Purchase_Frequency', 'Purchase_Recency_Group', 'Purchase_Frequency_Group', 'Service_Call_Group', 'Cart_Abandonment_Group', 'Estimated_Purchase_Value', 'Cart_Recency_Interaction', 'Low_Engagement_Recency']


## Section 06 — Train / Validation / Test Split

In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)
split_indices = {
    "train_indices": X_train.index.tolist(),
    "validation_indices": X_valid.index.tolist(),
    "test_indices": X_test.index.tolist(),
}

all_indices = set(X.index)

train_indices = set(X_train.index)
valid_indices = set(X_valid.index)
test_indices = set(X_test.index)

assert train_indices.isdisjoint(valid_indices)
assert train_indices.isdisjoint(test_indices)
assert valid_indices.isdisjoint(test_indices)

assert train_indices | valid_indices | test_indices == all_indices
    
assert list(X_train.columns) == list(X_valid.columns), \
    "Train and validation feature schemas do not match."

assert list(X_train.columns) == list(X_test.columns), \
    "Train and test feature schemas do not match."

print(f"X_train shape: {X_train.shape}")
print(f"X_valid shape: {X_valid.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_valid shape: {y_valid.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nTrain target distribution: {y_train.value_counts(normalize=True).to_dict()}")
print(f"Validation target distribution: {y_valid.value_counts(normalize=True).to_dict()}")
print(f"Test target distribution: {y_test.value_counts(normalize=True).to_dict()}")

SPLIT_PATH = MODELS_DIR / "data_split_indices.json"
with open(SPLIT_PATH, "w", encoding="utf-8") as split_file:
    json.dump(split_indices, split_file, indent=2)
print(MODELS_DIR / "data_split_indices.json")

assert len(X_train) > 0 and len(X_valid) > 0 and len(X_test) > 0

X_train shape: (35000, 32)
X_valid shape: (7500, 32)
X_test shape: (7500, 32)
y_train shape: (35000,)
y_valid shape: (7500,)
y_test shape: (7500,)

Train target distribution: {0: 0.711, 1: 0.289}
Validation target distribution: {0: 0.7109333333333333, 1: 0.2890666666666667}
Test target distribution: {0: 0.7110666666666666, 1: 0.2889333333333333}
D:\Desktop\Projects\customer-churn-analytics\models\preprocessor\data_split_indices.json


## Section 07 — Define Numerical and Categorical Features

In [7]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Total features: {len(numeric_features) + len(categorical_features)}")
print(f"Numerical features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

assert set(numeric_features).isdisjoint(categorical_features)
assert set(numeric_features + categorical_features) == set(X_train.columns)
assert list(X_train.columns) == list(X_valid.columns) == list(X_test.columns)



Total features: 32
Numerical features (25): ['Age', 'Membership_Years', 'Login_Frequency', 'Session_Duration_Avg', 'Pages_Per_Session', 'Cart_Abandonment_Rate', 'Wishlist_Items', 'Total_Purchases', 'Average_Order_Value', 'Days_Since_Last_Purchase', 'Discount_Usage_Rate', 'Returns_Rate', 'Email_Open_Rate', 'Customer_Service_Calls', 'Product_Reviews_Written', 'Social_Media_Engagement_Score', 'Mobile_App_Usage', 'Payment_Method_Diversity', 'Lifetime_Value', 'Credit_Balance', 'Engagement_Score', 'Purchase_Frequency', 'Estimated_Purchase_Value', 'Cart_Recency_Interaction', 'Low_Engagement_Recency']
Categorical features (7): ['Gender', 'Country', 'Signup_Quarter', 'Purchase_Recency_Group', 'Purchase_Frequency_Group', 'Service_Call_Group', 'Cart_Abandonment_Group']


## Section 08 — Build Preprocessing Pipeline

Numerical features đi qua `Median Imputation` + `StandardScaler`.
Categorical features đi qua `Most Frequent Imputation` + `OneHotEncoder(handle_unknown="ignore")`.
Sau đó, `ColumnTransformer` ghép hai nhóm thành output ML-ready, và toàn bộ parameters đều chỉ được fit trên `X_train`.

In [8]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

print("Preprocessor created:")
print(preprocessor)

Preprocessor created:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Membership_Years', 'Login_Frequency',
                                  'Session_Duration_Avg', 'Pages_Per_Session',
                                  'Cart_Abandonment_Rate', 'Wishlist_Items',
                                  'Total_Purchases', 'Average_Order_Value',
                                  'Days_Since_Last_Purchase',
                                  'Discount_Usage_Rate', 'Re...
                                  'Estimated_Purchase_Value',
                                  'Cart_Recency_Interaction',
                                  'Low_Engagement_Recency']),
                                ('cat',
                                 Pipeline(steps

## Section 09 — Fit and transform

In [9]:
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_valid_processed shape: {X_valid_processed.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")

processed_feature_names = preprocessor.get_feature_names_out().tolist()

print(f"Processed feature count: {len(processed_feature_names)}")
print("\nProcessed feature names:")
print(processed_feature_names)

assert not np.isnan(X_train_processed).any(), "NaN remains in train processed data."
assert not np.isnan(X_valid_processed).any(), "NaN remains in validation processed data."
assert not np.isnan(X_test_processed).any(), "NaN remains in test processed data."
assert X_train_processed.shape[1] == X_valid_processed.shape[1] == X_test_processed.shape[1]
assert X_train_processed.shape[0] == len(y_train)
assert X_valid_processed.shape[0] == len(y_valid)
assert X_test_processed.shape[0] == len(y_test)

X_train_processed shape: (35000, 54)
X_valid_processed shape: (7500, 54)
X_test_processed shape: (7500, 54)
Processed feature count: 54

Processed feature names:
['num__Age', 'num__Membership_Years', 'num__Login_Frequency', 'num__Session_Duration_Avg', 'num__Pages_Per_Session', 'num__Cart_Abandonment_Rate', 'num__Wishlist_Items', 'num__Total_Purchases', 'num__Average_Order_Value', 'num__Days_Since_Last_Purchase', 'num__Discount_Usage_Rate', 'num__Returns_Rate', 'num__Email_Open_Rate', 'num__Customer_Service_Calls', 'num__Product_Reviews_Written', 'num__Social_Media_Engagement_Score', 'num__Mobile_App_Usage', 'num__Payment_Method_Diversity', 'num__Lifetime_Value', 'num__Credit_Balance', 'num__Engagement_Score', 'num__Purchase_Frequency', 'num__Estimated_Purchase_Value', 'num__Cart_Recency_Interaction', 'num__Low_Engagement_Recency', 'cat__Gender_Female', 'cat__Gender_Male', 'cat__Gender_Other', 'cat__Country_Australia', 'cat__Country_Canada', 'cat__Country_France', 'cat__Country_Germany

## Section 10 — Validation and export


In [10]:
processed_feature_count = X_train_processed.shape[1]
feature_columns_consistent = (
    list(X_train.columns) == list(X_valid.columns) == list(X_test.columns)
)
metadata = {
    "dataset_path": str(DATA_PATH),

    "target": TARGET,

    "random_state": RANDOM_STATE,

    "split_ratios": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },

    "train_size": int(len(X_train)),
    "valid_size": int(len(X_valid)),
    "test_size": int(len(X_test)),

    "original_feature_count": int(len(approved_features)),
    "processed_feature_count": int(processed_feature_count),

    "numerical_feature_count": len(numeric_features),
    "categorical_feature_count": len(categorical_features),

    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "approved_features": approved_features,

    "excluded_features": excluded_features,
    "forbidden_features": FORBIDDEN_FEATURES,

    "preprocessing_config": {
        "numeric_imputation": "median",
        "numeric_scaling": "StandardScaler",
        "categorical_imputation": "most_frequent",
        "categorical_encoding": "OneHotEncoder",
        "onehot_handle_unknown": "ignore",
    },

    "processed_feature_names": processed_feature_names,
    "feature_columns_consistent": feature_columns_consistent,
}

print("\nPreprocessing validation summary:")
print(f"Original feature count: {metadata['original_feature_count']}")
print(f"Processed feature count: {metadata['processed_feature_count']}")
print(f"Train size: {metadata['train_size']}")
print(f"Validation size: {metadata['valid_size']}")
print(f"Test size: {metadata['test_size']}")
print(f"Target values preserved: {sorted(y_train.unique().tolist())}")
print(f"Feature names exported: {len(processed_feature_names)}")
print(f"All processed arrays have same column count: {X_train_processed.shape[1] == X_valid_processed.shape[1] == X_test_processed.shape[1]}")

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")
with open(MODELS_DIR / "preprocessor_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("\nSaved artifacts:")
print(MODELS_DIR / "preprocessor.joblib")
print(MODELS_DIR / "preprocessor_metadata.json")

assert sorted(y_train.unique().tolist()) == [0, 1]
assert set(y_valid.unique().tolist()).issubset({0, 1})
assert set(y_test.unique().tolist()).issubset({0, 1})
assert X_train_processed.shape[1] == X_valid_processed.shape[1] == X_test_processed.shape[1]
assert len(processed_feature_names) == processed_feature_count
assert metadata["feature_columns_consistent"]


Preprocessing validation summary:
Original feature count: 32
Processed feature count: 54
Train size: 35000
Validation size: 7500
Test size: 7500
Target values preserved: [0, 1]
Feature names exported: 54
All processed arrays have same column count: True

Saved artifacts:
D:\Desktop\Projects\customer-churn-analytics\models\preprocessor\preprocessor.joblib
D:\Desktop\Projects\customer-churn-analytics\models\preprocessor\preprocessor_metadata.json
